# ADRE-LoRA: Evidence-Grounded LoRA Fine-Tuning

**What this notebook does (full ADRE pipeline):**

1. Extract image evidence for ALL training studies (DenseNet121 classifier, zone crops, image features)
2. Retrieve relevant radiology guidelines per study (BM25 keyword matching over Fleischner/ACR inline knowledge base)
3. Build ADRE-style training prompts: `indication + image_evidence + guidelines → findings + impression`
4. Fine-tune a small LLM (Qwen2.5-3B-Instruct) with QLoRA on these rich prompts
5. Run inference on eval studies using the same ADRE context
6. Evaluate: BLEU-1/2/3/4, ROUGE-L, METEOR, BERTScore

**Key difference from notebook 02:** The LLM is trained to USE evidence and guidelines, not just free-text indication.

**Kaggle setup:**
- Attach dataset: `raddar/chest-xrays-indiana-university`
- GPU: T4 x1 (or P100)
- Internet: ON (to download model weights)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

pip("torchxrayvision")
pip("rank_bm25")
pip("transformers==4.46.3", "peft==0.13.2", "trl==0.11.4",
    "accelerate==1.2.1", "datasets==3.2.0")
pip("bitsandbytes")
pip("evaluate", "sacrebleu", "rouge-score", "nltk", "bert-score")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Dependencies installed.")

In [ ]:
# ── Cell 2: Dataset paths ──────────────────────────────────────────────────────
import os
from pathlib import Path

DATASET_ROOTS = [
    "/kaggle/input/chest-xrays-indiana-university",
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university",
    "/kaggle/input/raddar/chest-xrays-indiana-university",
]
DATA_ROOT = next((p for p in DATASET_ROOTS if Path(p).exists()), None)
if DATA_ROOT is None:
    raise RuntimeError("Could not find IU X-Ray dataset. Attach 'raddar/chest-xrays-indiana-university'.")
DATA_ROOT = Path(DATA_ROOT)
print(f"Dataset: {DATA_ROOT}")

IMG_DIR = DATA_ROOT / "images" / "images_normalized"
REPORTS_CSV = DATA_ROOT / "indiana_reports.csv"
PROJECTIONS_CSV = DATA_ROOT / "indiana_projections.csv"

OUT_DIR = Path("/kaggle/working")
EV_DIR  = OUT_DIR / "evidence"
EV_DIR.mkdir(parents=True, exist_ok=True)
print("Image dir exists:", IMG_DIR.exists())

In [ ]:
# ── Cell 3: Load and parse reports ─────────────────────────────────────────────
import re
import pandas as pd

reports_df = pd.read_csv(REPORTS_CSV)
proj_df    = pd.read_csv(PROJECTIONS_CSV)

# keep frontal (PA/AP) images only
frontal = proj_df[proj_df["projection"].str.upper().isin(["FRONTAL", "PA", "AP"])].copy()
frontal = frontal.drop_duplicates("uid").set_index("uid")

def clean(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"XXXX", "", text)
    return text.strip()

records = []
for _, row in reports_df.iterrows():
    uid = row.get("uid")
    findings  = clean(row.get("findings", ""))
    impression= clean(row.get("impression", ""))
    indication= clean(row.get("indication", ""))
    if not findings or not impression: continue
    if uid not in frontal.index: continue
    filename = frontal.loc[uid, "filename"]
    img_path = IMG_DIR / filename
    if not img_path.exists(): continue
    records.append(dict(uid=str(uid), indication=indication,
                        findings=findings, impression=impression,
                        image_path=str(img_path)))

print(f"Usable records: {len(records)}")

# deterministic 90/10 split (same seed as rest of project)
import random
random.seed(42)
random.shuffle(records)
split_n   = int(0.10 * len(records))
eval_data  = records[:split_n]
train_data = records[split_n:]
print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")

In [ ]:
# ── Cell 4: NeuralInference (DenseNet121) ──────────────────────────────────────
import torch
import numpy as np
import torchxrayvision as xrv
import torchvision.transforms as T
from PIL import Image

class NeuralInference:
    def __init__(self, weights="densenet121-res224-nih"):
        self.model = xrv.models.DenseNet(weights=weights)
        self.model.eval()
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        self.pathologies = self.model.pathologies

    def _prepare(self, img: np.ndarray) -> torch.Tensor:
        img = xrv.datasets.normalize(img, 255)
        transform = T.Compose([
            xrv.datasets.XRayCenterCrop(),
            xrv.datasets.XRayResizer(224),
        ])
        img = transform(img[None])          # (1, H, W)
        t = torch.from_numpy(img).unsqueeze(0).float()  # (1,1,H,W)
        if torch.cuda.is_available():
            t = t.cuda()
        return t

    def predict(self, img: np.ndarray) -> dict:
        t = self._prepare(img)
        with torch.no_grad():
            out = torch.sigmoid(self.model(t)).cpu().numpy()[0]
        return {p: float(s) for p, s in zip(self.pathologies, out)}

    def global_abnormality_score(self, probs: dict) -> float:
        return float(max(probs.values())) if probs else 0.0

    def regional_burden(self, zone_probs: dict) -> dict:
        out = {}
        for zone, probs in zone_probs.items():
            vals = [v for k, v in probs.items() if k]
            if vals:
                top_k = max(probs.items(), key=lambda x: x[1] if x[0] else 0)
                out[zone] = {
                    "max_pathology_prob": float(max(vals)),
                    "mean_pathology_prob": float(np.mean(vals)),
                    "top_pathology": top_k[0],
                }
        return out

print("NeuralInference defined.")

In [ ]:
# ── Cell 5: Image feature + quality helpers ────────────────────────────────────
import cv2

def load_image(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        pil = Image.open(path).convert("L")
        img = np.array(pil)
    img = img.astype(np.float32)
    return img

def zone_crop(img, row, col, rows=3, cols=2):
    h, w = img.shape[:2]
    rh, cw = h // rows, w // cols
    r0, r1 = row * rh, min((row + 1) * rh, h)
    c0, c1 = col * cw, min((col + 1) * cw, w)
    return img[r0:r1, c0:c1]

def extract_features(img):
    norm = img / 255.0 if img.max() > 1 else img
    h, w = norm.shape[:2]
    left  = norm[:, :w//2]
    right = norm[:, w//2:]
    sym = 1.0 - abs(float(left.mean()) - float(right.mean()))
    ctr_proxy = float(norm[h//3:2*h//3, w//3:2*w//3].mean())
    return {
        "opacity_density": float(norm.mean()),
        "symmetry_index":  sym,
        "ctr_proxy":       ctr_proxy,
    }

def assess_quality(img):
    norm = img / 255.0 if img.max() > 1 else img
    mean_val = float(norm.mean())
    h, w = norm.shape[:2]
    left_col  = float(norm[:, :5].mean())
    right_col = float(norm[:, -5:].mean())
    return {
        "exposure":              "adequate" if 0.2 < mean_val < 0.8 else "suboptimal",
        "inspiration_adequacy": "adequate" if h / w > 1.1 else "limited",
        "rotation_flag":        abs(left_col - right_col) > 0.05,
    }

print("Feature helpers defined.")

In [ ]:
# ── Cell 6: Batch evidence extraction ─────────────────────────────────────────
import json
from tqdm import tqdm

print("Loading DenseNet121 ...")
neural = NeuralInference()
print("Ready.")

def extract_evidence(uid: str, image_path: str, indication: str) -> dict:
    img = load_image(image_path)
    feats = extract_features(img)
    qual  = assess_quality(img)
    
    global_probs = neural.predict(img)
    global_score = neural.global_abnormality_score(global_probs)
    
    zone_probs = {}
    for r in range(3):
        for c in range(2):
            crop = zone_crop(img, r, c)
            zone_key = f"zone_{['upper','mid','lower'][r]}_{['left','right'][c]}"
            zone_probs[zone_key] = neural.predict(crop)
    regional = neural.regional_burden(zone_probs)
    
    return {
        "study_id":              uid,
        "patient_metadata":      {"indication": indication},
        "global_pathology_probs": global_probs,
        "global_abnormality_score": global_score,
        "regional_burden":       regional,
        "image_features":        feats,
        "quality_indicators":    qual,
    }

all_records = train_data + eval_data
print(f"Extracting evidence for {len(all_records)} studies ...")

for rec in tqdm(all_records):
    uid = rec["uid"]
    out_path = EV_DIR / f"{uid}.json"
    if out_path.exists():
        continue
    try:
        ev = extract_evidence(uid, rec["image_path"], rec["indication"])
        out_path.write_text(json.dumps(ev, indent=2), encoding="utf-8")
    except Exception as e:
        print(f"  skip {uid}: {e}")

print(f"Evidence files: {len(list(EV_DIR.glob('*.json')))}")

In [ ]:
# ── Cell 7: Inline radiology guidelines (BM25 retrieval) ──────────────────────
# Full inline knowledge base — no external index file needed.
# Key Fleischner Society + ACR CXR reporting guidelines.

from rank_bm25 import BM25Okapi

GUIDELINES_KB = [
    {"id": "fleischner_consolidation", "title": "Fleischner: Consolidation",
     "text": "Consolidation is a homogeneous increase in pulmonary parenchymal attenuation that obscures the margins of vessels and airway walls. Air bronchograms may be visible. Etiology includes pneumonia, hemorrhage, edema, and aspiration. The finding should be described by location (lobe, segment), extent, and associated findings such as effusion or lymphadenopathy."},
    {"id": "fleischner_atelectasis", "title": "Fleischner: Atelectasis",
     "text": "Atelectasis refers to diminished volume of the lung. On chest radiograph it appears as increased density with volume loss signs including displacement of fissures, elevation of hemidiaphragm, and mediastinal shift. Plate-like (discoid) atelectasis appears as a linear opacity parallel to the diaphragm, commonly at the lung base."},
    {"id": "fleischner_effusion", "title": "Fleischner: Pleural Effusion",
     "text": "Pleural effusion appears as blunting of the costophrenic angle on PA radiograph, requiring approximately 200 mL to be visible. Larger effusions produce a meniscus sign with homogeneous basal opacity. Bilateral effusions are commonly due to cardiac failure or hypoalbuminemia. Unilateral effusions require further investigation for exudative causes."},
    {"id": "fleischner_pneumothorax", "title": "Fleischner: Pneumothorax",
     "text": "Pneumothorax is seen as a visible pleural line with absence of lung markings peripheral to it. On supine radiographs a deep sulcus sign may be the only finding. Tension pneumothorax causes mediastinal shift away from the affected side and requires emergency decompression. Size is estimated as percentage of hemithorax involved."},
    {"id": "fleischner_nodule", "title": "Fleischner: Pulmonary Nodule",
     "text": "A pulmonary nodule is a rounded or irregular opacity, well or poorly defined, measuring up to 3 cm. Nodules smaller than 6 mm in low-risk patients require no follow-up. Larger nodules or those with spiculated margins require CT characterization. Calcification patterns (central, laminar, popcorn) suggest benignity."},
    {"id": "fleischner_cardiomegaly", "title": "Fleischner: Cardiomegaly",
     "text": "Cardiomegaly is defined as a cardiothoracic ratio (CTR) greater than 0.5 on a PA radiograph. It may indicate dilated cardiomyopathy, pericardial effusion, or volume overload. Associated findings include pulmonary vascular congestion, Kerley B lines, and pleural effusion in cardiac failure."},
    {"id": "fleischner_edema", "title": "Fleischner: Pulmonary Edema",
     "text": "Pulmonary edema on CXR shows bilateral perihilar haziness (bat-wing pattern), vascular redistribution to upper lobes, peribronchial cuffing, Kerley B lines at the bases, and pleural effusion. Cardiogenic edema is typically bilateral and symmetric. Non-cardiogenic (ARDS) may be more patchy and peripheral."},
    {"id": "fleischner_fibrosis", "title": "Fleischner: Pulmonary Fibrosis",
     "text": "Pulmonary fibrosis appears as reticular or reticulonodular opacities, predominantly at the lung bases and periphery. Volume loss is common. Honeycomb pattern indicates advanced fibrosis. Traction bronchiectasis and subpleural involvement are characteristic of usual interstitial pneumonia (UIP) pattern."},
    {"id": "fleischner_emphysema", "title": "Fleischner: Emphysema",
     "text": "Emphysema on CXR presents with hyperinflation (flattened diaphragms, increased AP diameter), increased retrosternal airspace, and decreased vascular markings. Bullae may be visible as well-defined avascular regions. Paraseptal emphysema appears as subpleural lucencies. Correlation with clinical history and pulmonary function tests is important."},
    {"id": "acr_normal", "title": "ACR: Normal Chest Radiograph",
     "text": "A normal chest radiograph demonstrates clear lung fields without focal opacity, effusion, or pneumothorax. The cardiac silhouette should be normal in size (CTR less than 0.5 on PA). The mediastinum should be of normal width. The diaphragms should be smooth and well-defined. Bony structures should be intact. Report as: No acute cardiopulmonary abnormality."},
    {"id": "acr_reporting", "title": "ACR: CXR Reporting Standards",
     "text": "A complete chest radiograph report should include: (1) Lung parenchyma — describe any opacity, its location (upper/mid/lower zone, left/right), pattern (consolidation, atelectasis, interstitial), and extent. (2) Pleura — effusion, pneumothorax, thickening. (3) Cardiac — size, contour. (4) Mediastinum — width, tracheal deviation, hilar enlargement. (5) Osseous structures — rib fractures, vertebral anomalies. (6) Soft tissues. Impression should summarize key findings and clinical significance."},
    {"id": "acr_pleural_thickening", "title": "ACR: Pleural Thickening",
     "text": "Pleural thickening appears as a smooth or irregular increased density along the pleural surface. It is commonly due to prior inflammatory disease, asbestos exposure, or resolved effusion. Significant bilateral pleural thickening may restrict lung expansion. Calcified plaques are characteristic of asbestos exposure."},
    {"id": "acr_mass", "title": "ACR: Pulmonary Mass",
     "text": "A pulmonary mass is a rounded opacity greater than 3 cm. Malignancy must be excluded. Important features include border characteristics (spiculated vs smooth), internal density, satellite nodules, and associated mediastinal lymphadenopathy. CT evaluation is mandatory for all masses identified on plain radiograph."},
    {"id": "acr_infiltrate", "title": "ACR: Infiltration / Airspace Disease",
     "text": "Infiltration or airspace disease describes a heterogeneous increase in lung density, often with poorly defined margins. This may represent early pneumonia, aspiration, or non-specific inflammatory process. The distribution (lobar, segmental, diffuse) helps narrow the differential. Associated air bronchograms suggest alveolar filling process."},
    {"id": "acr_hernia", "title": "ACR: Hiatal / Diaphragmatic Hernia",
     "text": "Hiatal hernia appears as a retrocardiac mass with or without an air-fluid level on the lateral view. Large hernias may be visible on PA as a mass behind the cardiac shadow. Diaphragmatic hernia may show bowel loops in the thorax. Most hiatal hernias are incidental findings."},
]

# Build BM25 index over guidelines
_tokenized = [g["text"].lower().split() for g in GUIDELINES_KB]
_bm25 = BM25Okapi(_tokenized)

def retrieve_guidelines(query: str, top_k: int = 4) -> list[dict]:
    tokens = query.lower().split()
    scores = _bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [GUIDELINES_KB[i] for i in top_idx]

# Quick test
test_chunks = retrieve_guidelines("consolidation pneumonia left lower lobe")
print("Top retrieved guidelines for test query:")
for c in test_chunks:
    print(" ", c["title"])

In [ ]:
# ── Cell 8: ADRE prompt builder ────────────────────────────────────────────────

def fmt_pathology(probs: dict, top_n: int = 10) -> str:
    items = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:top_n]
    lines = []
    for label, score in items:
        if not label: continue
        flag = "HIGH" if score > 0.65 else ("MOD" if score > 0.55 else "low")
        lines.append(f"  {label:<25} {score:.3f}  [{flag}]")
    return "\n".join(lines) or "  (not available)"

def fmt_regional(regional: dict) -> str:
    order = ["zone_upper_left","zone_upper_right","zone_mid_left",
             "zone_mid_right","zone_lower_left","zone_lower_right"]
    lines = []
    for zone in order:
        info = regional.get(zone, {})
        if not info: continue
        label = zone.replace("zone_","").replace("_"," ").title()
        lines.append(f"  {label:<20} top={info.get('top_pathology','?')} "
                     f"(score={info.get('max_pathology_prob',0):.3f})")
    return "\n".join(lines) or "  (not available)"

def fmt_features(feats: dict) -> str:
    return (f"  Opacity density:  {feats.get('opacity_density',0):.3f}\n"
            f"  Symmetry index:   {feats.get('symmetry_index',0):.3f}\n"
            f"  CTR proxy:        {feats.get('ctr_proxy',0):.3f}")

def fmt_quality(qual: dict) -> str:
    return (f"  Exposure:         {qual.get('exposure','?')}\n"
            f"  Inspiration:      {qual.get('inspiration_adequacy','?')}\n"
            f"  Rotation:         {'present' if qual.get('rotation_flag') else 'minimal'}")

def fmt_guidelines(chunks: list) -> str:
    parts = []
    for c in chunks:
        parts.append(f"[{c['title']}]\n{c['text'][:350]}")
    return "\n\n".join(parts) or "(none retrieved)"

def build_adre_prompt(indication: str, evidence: dict) -> str:
    """Build the full ADRE-style input prompt."""
    # Retrieve guidelines based on indication + top pathologies
    top_paths = sorted(evidence.get("global_pathology_probs",{}).items(),
                       key=lambda x: x[1], reverse=True)[:5]
    query = indication + " " + " ".join(p for p, _ in top_paths if p)
    chunks = retrieve_guidelines(query, top_k=4)

    return f"""You are a board-certified radiologist. You have access to the patient's clinical indication, AI-extracted image evidence, and relevant radiology guidelines. Use ALL of this information to write a structured chest X-ray report.

=== CLINICAL INDICATION ===
{indication or 'Not provided'}

=== AI IMAGE EVIDENCE ===

Pathology Scores (DenseNet121 classifier, 0=normal 1=abnormal):
{fmt_pathology(evidence.get('global_pathology_probs', {}))}

Global Abnormality Score: {evidence.get('global_abnormality_score', 0.0):.3f}

Regional Burden by Lung Zone:
{fmt_regional(evidence.get('regional_burden', {}))}

Image Features:
{fmt_features(evidence.get('image_features', {}))}

Quality Indicators:
{fmt_quality(evidence.get('quality_indicators', {}))}

=== RELEVANT RADIOLOGY GUIDELINES ===
{fmt_guidelines(chunks)}

=== TASK ===
Write the report with exactly these two sections:

FINDINGS:
[Detailed paragraph covering lungs (upper/mid/lower zones, left and right), heart size, mediastinum, pleura, and bones.]

IMPRESSION:
[1-3 sentence clinical summary.]"""

# Test one prompt
test_ev = json.loads((EV_DIR / (train_data[0]["uid"] + ".json")).read_text())
test_prompt = build_adre_prompt(train_data[0]["indication"], test_ev)
print("Sample prompt preview (first 600 chars):")
print(test_prompt[:600])
print(f"\nFull prompt length: {len(test_prompt)} chars")

In [ ]:
# ── Cell 9: Build HuggingFace datasets ────────────────────────────────────────
from datasets import Dataset

def build_messages(rec: dict) -> dict | None:
    uid = rec["uid"]
    ev_path = EV_DIR / f"{uid}.json"
    if not ev_path.exists():
        return None
    evidence = json.loads(ev_path.read_text())
    prompt = build_adre_prompt(rec["indication"], evidence)
    target = f"FINDINGS:\n{rec['findings'].strip()}\n\nIMPRESSION:\n{rec['impression'].strip()}"
    return {
        "uid": uid,
        "messages": [
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": target},
        ]
    }

print("Building training dataset ...")
train_rows = [r for rec in train_data if (r := build_messages(rec)) is not None]
eval_rows  = [r for rec in eval_data  if (r := build_messages(rec)) is not None]

train_ds = Dataset.from_list(train_rows)
eval_ds  = Dataset.from_list(eval_rows)
print(f"Train examples: {len(train_ds)}  |  Eval examples: {len(eval_ds)}")

# Save eval rows for later reconstruction
with open("/kaggle/working/adre_eval_records.jsonl", "w") as f:
    for r in eval_rows:
        f.write(json.dumps({"uid": r["uid"]}) + "\n")

In [ ]:
# ── Cell 10: Load model + LoRA config ─────────────────────────────────────────
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# Clean previous state
for name in ["model", "tokenizer", "trainer"]:
    if name in globals(): del globals()[name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# QLoRA 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID} ...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Loaded with 4-bit quantization.")
except Exception as e:
    print(f"4-bit failed ({e}), falling back to fp16 ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# LoRA config — full 7-projection coverage (best in prior experiments)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("LoRA config: r=16, full 7-projection coverage.")

In [ ]:
# ── Cell 11: SFT Training ──────────────────────────────────────────────────────

sft_config = SFTConfig(
    output_dir="/kaggle/working/adre_lora_ckpt",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    save_strategy="epoch",
    max_seq_length=2048,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Starting ADRE-LoRA training ...")
print(f"  Training examples: {len(train_ds)}")
print(f"  Epochs: 3  |  Effective batch: {1 * 8}")
print(f"  Max seq length: 2048 (ADRE prompts are ~1200-1600 tokens)")

trainer.train()
trainer.save_model("/kaggle/working/adre_lora_adapter")
print("Training complete. Adapter saved.")

In [ ]:
# ── Cell 12: ADRE-style inference on eval set ──────────────────────────────────

def clean_output(text: str) -> str:
    """Remove tokenizer dialogue noise, keep FINDINGS+IMPRESSION only."""
    for marker in ["<|im_end|>", "<|endoftext|>", "</s>",
                   "\nUser:", "\nuser:", "\nassistant:", "\nHuman:"]:
        if marker in text:
            text = text[:text.index(marker)]
    return text.strip()

adre_lora_model = trainer.model
adre_lora_model.eval()

predictions = []
print(f"Running ADRE-LoRA inference on {len(eval_rows)} eval studies ...")

for row in tqdm(eval_rows):
    uid = row["uid"]
    user_prompt = row["messages"][0]["content"]  # full ADRE prompt
    reference   = row["messages"][1]["content"]  # ground truth

    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(adre_lora_model.device)

    with torch.no_grad():
        out = adre_lora_model.generate(
            **inputs,
            max_new_tokens=300,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            min_new_tokens=60,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    generated = tokenizer.decode(gen_ids, skip_special_tokens=True)
    generated = clean_output(generated)

    predictions.append({
        "uid": uid,
        "generated": generated,
        "reference": reference,
    })

# Save predictions
pred_path = "/kaggle/working/adre_lora_predictions.jsonl"
with open(pred_path, "w") as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Saved {len(predictions)} predictions to {pred_path}")
print("\nSample generated report (first 400 chars):")
print(predictions[0]["generated"][:400])

In [ ]:
# ── Cell 13: Evaluation ────────────────────────────────────────────────────────
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer as rouge_lib
from bert_score import score as bert_score_fn
import nltk

smoother = SmoothingFunction().method1
rouge = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

refs_tok  = [nltk.word_tokenize(p["reference"].lower())  for p in predictions]
hyps_tok  = [nltk.word_tokenize(p["generated"].lower())  for p in predictions]

bleu1 = corpus_bleu([[r] for r in refs_tok], hyps_tok,
                    weights=(1,0,0,0), smoothing_function=smoother)
bleu2 = corpus_bleu([[r] for r in refs_tok], hyps_tok,
                    weights=(.5,.5,0,0), smoothing_function=smoother)
bleu3 = corpus_bleu([[r] for r in refs_tok], hyps_tok,
                    weights=(.33,.33,.34,0), smoothing_function=smoother)
bleu4 = corpus_bleu([[r] for r in refs_tok], hyps_tok,
                    weights=(.25,.25,.25,.25), smoothing_function=smoother)

rouge_l = float(np.mean([
    rouge.score(p["reference"], p["generated"])["rougeL"].fmeasure
    for p in predictions
]))

meteor = float(np.mean([
    meteor_score([r], h) for r, h in zip(refs_tok, hyps_tok)
]))

print("Computing BERTScore (this takes ~1-2 min) ...")
P, R, F1 = bert_score_fn(
    [p["generated"] for p in predictions],
    [p["reference"]  for p in predictions],
    lang="en", model_type="roberta-large", verbose=False,
)
bertscore_f1 = float(F1.mean())

metrics = {
    "setting": "adre_lora",
    "n": len(predictions),
    "bleu1": round(bleu1, 4),
    "bleu2": round(bleu2, 4),
    "bleu3": round(bleu3, 4),
    "bleu4": round(bleu4, 4),
    "rouge_l": round(rouge_l, 4),
    "meteor": round(meteor, 4),
    "bertscore_f1": round(bertscore_f1, 4),
}

with open("/kaggle/working/adre_lora_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "="*50)
print("ADRE-LoRA Results (Evidence-Grounded LoRA):")
print("="*50)
for k, v in metrics.items():
    if k not in ("setting", "n"):
        print(f"  {k:<20} {v:.4f}")
print(f"  n = {metrics['n']}")

In [ ]:
# ── Cell 14: Zip outputs for download ─────────────────────────────────────────
import shutil

shutil.make_archive("/kaggle/working/adre_lora_outputs", "zip",
                    "/kaggle/working",
                    base_dir=None)

# List output files
for f in sorted(Path("/kaggle/working").glob("*.json*")):
    print(f.name, f.stat().st_size // 1024, "KB")

print("\nDone! Download: adre_lora_metrics.json + adre_lora_predictions.jsonl")